# Single-PCA Per-Epoch Encoding Drift — SLURM Orchestrator

Fits Poisson GLM encoding models on 2-hour epoch windows using a **shared global
PCA basis** (fit on all words from all epochs combined).  All per-epoch weight
vectors live in the same feature space, making cosine-distance comparisons
across epochs — including within the same day — interpretable.

**Phase 0** (one SLURM job per patient):  
Fits a single PCA preprocessing bundle on ALL words from ALL 2-hour epochs across
ALL recording days.

**Phase 1** (one SLURM job per (patient, date, epoch)):  
Fit an epoch-level GLM using the global bundle and alpha hyperparameters from the
existing nested-CV per-epoch results.

No cross-epoch testing (Phase 2) — the W matrices are the output; cosine distances
between them are computed in figure_6.ipynb (Panel F).

In [22]:
from pathlib import Path
import subprocess

import dill as pickle
import numpy as np
import pandas as pd

In [23]:
# ── Paths ──────────────────────────────────────────────────────────────────────
PROJ_DIR  = Path('/mnt/labworlds/Hayden/Hayden_Lab/speech_247')
VAD_ROOT  = PROJ_DIR / 'vad_new'

WORKER_BUNDLE = Path('/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!'
                     '/functional_drift/encoding_singlepca_epoch_bundle_worker.py')
WORKER_TRAIN  = Path('/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!'
                     '/functional_drift/encoding_singlepca_epoch_train_worker.py')
PYTHON_BIN    = '/scratch/tahaismail424/miniforge3/envs/speech_247_env/bin/python3'

PATIENTS       = None   # None => scan all Y* folders
FORCE_RESUBMIT = False

# ── Source run (per-epoch nested CV results) ──────────────────────────────────
SOURCE_RUN   = 'word_level_duration_cv_filtered_speech_per_epoch'
DRIFT_SUBDIR = 'encoding_singlepca_epoch_filtered'

# ── Model params (must match the source run) ──────────────────────────────────
SPIKE_OFFSET_IDX = 0
GPT2_LAYER       = -1
N_PCA            = 100

# ── SLURM — Phase 0 (CPU, fits global PCA) ────────────────────────────────────
CONDA_ACTIVATE = 'source /scratch/tahaismail424/.bash_profile && conda activate speech_247_env'

P0_PARTITION = 'Universal'
P0_GRES      = ''
P0_CPUS      = 8
P0_MEM       = '64G'
P0_TIME      = '04:00:00'

# ── SLURM — Phase 1 (CPU, fits per-epoch GLM) ─────────────────────────────────
P1_PARTITION = 'Universal'
P1_GRES      = ''
P1_CPUS      = 16
P1_MEM       = '32G'
P1_TIME      = '48:00:00'

LOGS_DIR    = VAD_ROOT / f'{DRIFT_SUBDIR}_slurm_logs'
SCRIPTS_DIR = VAD_ROOT / f'{DRIFT_SUBDIR}_slurm_scripts'
LOGS_DIR.mkdir(parents=True, exist_ok=True)
SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

print('source run: ', SOURCE_RUN)
print('drift dir:  ', DRIFT_SUBDIR)

source run:  word_level_duration_cv_filtered_speech_per_epoch
drift dir:   encoding_singlepca_epoch_filtered


## 1. Discover Patients and Per-Epoch Model Results

In [24]:
def first_existing(paths):
    for p in paths:
        if p is not None and Path(p).exists():
            return Path(p)
    return None


def resolve_patient_inputs(patient):
    r = VAD_ROOT / patient
    emb = first_existing([r / 'embeddings' / f'{patient}_gpt2_embeddings.npy',
                           r / 'all_convo_recording' / 'all_words_filtered_all_layers_gpt2.npy'])
    cnt = first_existing([r / 'neural_embeddings' / 'word_spike_counts_offsets_all.npy',
                           r / 'all_convo_recording' / 'word_spike_counts_offsets_all.npy'])
    dur = first_existing([r / 'neural_embeddings' / 'word_durs.npy',
                           r / 'all_convo_recording' / 'word_durs.npy'])
    return emb, cnt, dur


def get_epoch_model_entries(patient):
    """
    Returns list of (date_str, epoch_str, cv_pkl_path, word_idx_path) for epochs
    that have completed nested-CV results (pkl + tar + word_idx + SUCCESS).
    """
    base = VAD_ROOT / patient / 'encoding' / SOURCE_RUN
    if not base.exists():
        return []
    entries = []
    for date_dir in sorted(base.iterdir()):
        if not date_dir.is_dir():
            continue
        for epoch_dir in sorted(date_dir.iterdir()):
            if not epoch_dir.is_dir():
                continue
            cv_pkl   = epoch_dir / f'{patient}_encoding_results_cv.pkl'
            cv_tar   = epoch_dir / f'{patient}_encoding_models_cv.tar'
            success  = epoch_dir / f'{patient}_SUCCESS'
            idx_files = list(epoch_dir.glob(f'{patient}_*_word_idx.npy'))
            if cv_pkl.exists() and cv_tar.exists() and success.exists() and idx_files:
                entries.append((date_dir.name, epoch_dir.name, cv_pkl, idx_files[0]))
    return entries


if PATIENTS is None:
    all_patients = sorted(p.name for p in VAD_ROOT.iterdir()
                          if p.is_dir() and p.name.startswith('Y'))
else:
    all_patients = list(PATIENTS)

overview = []
for pt in all_patients:
    entries = get_epoch_model_entries(pt)
    n_dates = len(set(d for d,_,_,_ in entries))
    overview.append(dict(patient=pt, n_epochs=len(entries), n_dates=n_dates))

overview_df = pd.DataFrame(overview)
print(f'Patients with epoch results: {int((overview_df.n_epochs > 0).sum())}')
print(f'Total epoch entries: {int(overview_df.n_epochs.sum())}')
display(overview_df[overview_df.n_epochs > 0])

Patients with epoch results: 21
Total epoch entries: 863


,patient,n_epochs,n_dates
0,YEY,8,2
1,YEZ,25,6
2,YFA,54,9
3,YFB,48,9
4,YFC,50,9
5,YFD,33,6
6,YFE,20,4
7,YFF,58,10
8,YFG,22,5
9,YFI,39,7


## 2. Phase-0 Status and Submission

One job per patient — fits global PCA bundle on all words from all epochs.

In [25]:
def p0_paths(patient):
    base_dir = VAD_ROOT / patient / 'functional_drift' / DRIFT_SUBDIR
    return dict(
        base_dir   = base_dir,
        bundle_pkl = base_dir / f'{patient}_global_bundle.pkl',
        success    = base_dir / f'{patient}_GLOBAL_BUNDLE_SUCCESS',
        error      = base_dir / f'{patient}_global_bundle_error.txt',
    )


rows_p0 = []
for pt in all_patients:
    emb, cnt, dur = resolve_patient_inputs(pt)
    if any(p is None for p in [emb, cnt, dur]):
        continue
    if not get_epoch_model_entries(pt):
        continue
    pp   = p0_paths(pt)
    done = pp['success'].exists()
    err  = pp['error'].exists()
    rows_p0.append(dict(
        patient=pt,
        embeddings_path=emb, counts_path=cnt, durations_path=dur,
        **pp,
        done=done, has_error=err,
        needs_p0=not done or FORCE_RESUBMIT,
    ))

p0_df = pd.DataFrame(rows_p0)
print(f'Phase-0: {len(p0_df)} patients  |  done: {p0_df.done.sum()}  '
      f'|  errors: {p0_df.has_error.sum()}  |  needs submit: {p0_df.needs_p0.sum()}')
p0_df[['patient','done','has_error','needs_p0']]

KeyboardInterrupt: 

In [ ]:
DRY_RUN = False

submitted_p0 = {}
failed_p0    = []

for _, row in p0_df[p0_df['needs_p0']].iterrows():
    pt      = row['patient']
    out_dir = row['base_dir']
    out_dir.mkdir(parents=True, exist_ok=True)

    reset = f"rm -f {row['success']} {row['error']}" if FORCE_RESUBMIT else 'true'
    cmd = (
        f'{PYTHON_BIN} {WORKER_BUNDLE} {pt} {VAD_ROOT} {out_dir} '
        f'--source-run {SOURCE_RUN} '
        f'--n-pca {N_PCA} '
        f'--gpt2-layer {GPT2_LAYER} '
        f'--spike-offset-idx {SPIKE_OFFSET_IDX} '
        f'--embeddings-path {row["embeddings_path"]} '
        f'--counts-path {row["counts_path"]} '
        f'--durations-path {row["durations_path"]}'
    )
    script = '\n'.join([
        '#!/bin/bash',
        f'#SBATCH --job-name=spca_ep_bundle_{pt}',
        f'#SBATCH --partition={P0_PARTITION}',
        f'#SBATCH --cpus-per-task={P0_CPUS}',
        f'#SBATCH --qos=big_batch_tier',
        #f'#SBATCH --exclude=guppy-1',
        f'#SBATCH --mem={P0_MEM}',
        f'#SBATCH --time={P0_TIME}',
        f'#SBATCH --output={LOGS_DIR}/{pt}_p0_%j.out',
        f'#SBATCH --error={LOGS_DIR}/{pt}_p0_%j.err',
        '', 'set -eo pipefail', CONDA_ACTIVATE,
        'echo "HOST: $(hostname)  START: $(date)"', reset, cmd, 'echo "END: $(date)"',
    ])
    sbatch_path = SCRIPTS_DIR / f'{pt}_p0.sbatch'
    sbatch_path.write_text(script)

    if DRY_RUN:
        print(f'[DRY] P0: {pt}'); continue

    try:
        res = subprocess.run(['sbatch','--parsable',str(sbatch_path)],
                             capture_output=True, text=True, check=True)
        jid = res.stdout.strip()
        submitted_p0[pt] = jid
        print(f'submitted P0: {pt} -> {jid}')
    except subprocess.CalledProcessError as exc:
        failed_p0.append((pt, exc.stderr.strip()))
        print(f'FAILED P0: {pt}\n{exc.stderr}')

print(f'\nP0 submitted={len(submitted_p0)}, failed={len(failed_p0)}')

submitted P0: YFA -> 487494

P0 submitted=1, failed=0


## 3. Phase-1 Status and Submission

One job per (patient, date, epoch) — fits epoch-level GLM using global bundle.
Depends on Phase-0 bundle for the patient.

In [ ]:
def p1_paths(patient, date_str, epoch_str):
    drift_dir = VAD_ROOT / patient / 'functional_drift' / DRIFT_SUBDIR / date_str / epoch_str
    return dict(
        out_dir   = drift_dir,
        model_tar = drift_dir / f'{patient}_epoch_model.tar',
        train_idx = drift_dir / f'{patient}_epoch_train_idx.npy',
        meta_json = drift_dir / f'{patient}_epoch_meta.json',
        success   = drift_dir / f'{patient}_TRAIN_SUCCESS',
        error     = drift_dir / f'{patient}_error.txt',
    )


rows_p1 = []
for pt in all_patients:
    emb, cnt, dur = resolve_patient_inputs(pt)
    if any(p is None for p in [emb, cnt, dur]):
        continue
    pp0 = p0_paths(pt)
    for date_str, epoch_str, cv_pkl, word_idx_path in get_epoch_model_entries(pt):
        pp    = p1_paths(pt, date_str, epoch_str)
        done  = pp['success'].exists()
        error = pp['error'].exists()
        rows_p1.append(dict(
            patient=pt, date_str=date_str, epoch_str=epoch_str,
            cv_results_pkl=cv_pkl, word_idx_path=word_idx_path,
            embeddings_path=emb, counts_path=cnt, durations_path=dur,
            global_bundle_pkl=pp0['bundle_pkl'],
            **pp,
            p0_done=pp0['success'].exists(),
            p0_jid=submitted_p0.get(pt, None),
            done=done, has_error=error,
            needs_p1=not done or FORCE_RESUBMIT,
        ))

p1_df = pd.DataFrame(rows_p1)
print(f'Phase-1: {len(p1_df)} epoch jobs  |  done: {p1_df.done.sum()}  '
      f'|  errors: {p1_df.has_error.sum()}  |  P0 done: {p1_df.p0_done.sum()}  '
      f'|  needs submit: {p1_df.needs_p1.sum()}')
p1_df[['patient','date_str','epoch_str','done','has_error','p0_done','needs_p1']].head(20)

KeyboardInterrupt: 

In [26]:
DRY_RUN = False

submitted_p1 = {}
failed_p1    = []

for _, row in p1_df[p1_df['needs_p1']].iterrows():
    pt         = row['patient']
    date_str   = row['date_str']
    epoch_str  = row['epoch_str']
    out_dir    = row['out_dir']
    out_dir.mkdir(parents=True, exist_ok=True)

    if not row['p0_done'] and not row['p0_jid']:
        print(f'  skipping {pt} {date_str}/{epoch_str}: P0 not done and no pending P0 job')
        continue

    dep_flag  = f'--dependency=afterok:{row["p0_jid"]}' if (row['p0_jid'] and not row['p0_done']) else ''
    reset     = f"rm -f {row['success']} {row['error']}" if FORCE_RESUBMIT else 'true'
    gres_line = f'#SBATCH --gres={P1_GRES}' if P1_GRES else ''

    cmd = (
        f'{PYTHON_BIN} {WORKER_TRAIN} {pt} {VAD_ROOT} {out_dir} '
        f'--train-date {date_str} '
        f'--train-epoch {epoch_str} '
        f'--cv-results-pkl {row["cv_results_pkl"]} '
        f'--word-idx-path {row["word_idx_path"]} '
        f'--global-bundle-pkl {row["global_bundle_pkl"]} '
        f'--embeddings-path {row["embeddings_path"]} '
        f'--counts-path {row["counts_path"]} '
        f'--durations-path {row["durations_path"]} '
        f'--spike-offset-idx {SPIKE_OFFSET_IDX} '
        f'--gpt2-layer {GPT2_LAYER}'
    )

    script = '\n'.join([
        '#!/bin/bash',
        f'#SBATCH --job-name=spca_ep_train_{pt}_{date_str}_{epoch_str}',
        f'#SBATCH --partition={P1_PARTITION}',
        gres_line,
        f'#SBATCH --cpus-per-task={P1_CPUS}',
        f'#SBATCH --qos=big_batch_tier',
        #f'#SBATCH --exclude=guppy-1',
        f'#SBATCH --mem={P1_MEM}',
        f'#SBATCH --time={P1_TIME}',
        f'#SBATCH --output={LOGS_DIR}/{pt}_{date_str}_{epoch_str}_p1_%j.out',
        f'#SBATCH --error={LOGS_DIR}/{pt}_{date_str}_{epoch_str}_p1_%j.err',
        '', 'set -eo pipefail', CONDA_ACTIVATE,
        'echo "HOST: $(hostname)  START: $(date)"', reset, cmd, 'echo "END: $(date)"',
    ])

    sbatch_path = SCRIPTS_DIR / f'{pt}_{date_str}_{epoch_str}_p1.sbatch'
    sbatch_path.write_text(script)

    if DRY_RUN:
        print(f'[DRY] P1: {pt} {date_str}/{epoch_str}  dep={dep_flag}')
        continue

    try:
        sbatch_args = ['sbatch','--parsable']
        if dep_flag: sbatch_args.append(dep_flag)
        sbatch_args.append(str(sbatch_path))
        res = subprocess.run(sbatch_args, capture_output=True, text=True, check=True)
        jid = res.stdout.strip()
        submitted_p1[(pt, date_str, epoch_str)] = jid
        print(f'submitted P1: {pt} {date_str}/{epoch_str} -> {jid}')
    except subprocess.CalledProcessError as exc:
        failed_p1.append((pt, date_str, epoch_str, exc.stderr.strip()))
        print(f'FAILED P1: {pt} {date_str}/{epoch_str}\n{exc.stderr}')

print(f'\nP1 submitted={len(submitted_p1)}, failed={len(failed_p1)}')

submitted P1: YEZ 2024-04-09/15-17 -> 487495
submitted P1: YEZ 2024-04-09/17-19 -> 487496
submitted P1: YEZ 2024-04-09/19-21 -> 487497
submitted P1: YEZ 2024-04-09/21-23 -> 487498
submitted P1: YEZ 2024-04-10/09-11 -> 487499
submitted P1: YEZ 2024-04-10/11-13 -> 487500
submitted P1: YEZ 2024-04-10/13-15 -> 487501
submitted P1: YEZ 2024-04-10/15-17 -> 487502
submitted P1: YEZ 2024-04-11/13-15 -> 487503
submitted P1: YEZ 2024-04-11/19-21 -> 487504
submitted P1: YEZ 2024-04-11/21-23 -> 487505
submitted P1: YEZ 2024-04-12/09-11 -> 487506
submitted P1: YEZ 2024-04-12/11-13 -> 487507
submitted P1: YEZ 2024-04-12/13-15 -> 487508
submitted P1: YEZ 2024-04-12/15-17 -> 487509
submitted P1: YEZ 2024-04-14/11-13 -> 487510
submitted P1: YEZ 2024-04-14/13-15 -> 487511
submitted P1: YEZ 2024-04-14/15-17 -> 487512
submitted P1: YEZ 2024-04-14/17-19 -> 487513
submitted P1: YEZ 2024-04-14/19-21 -> 487514
submitted P1: YEZ 2024-04-14/21-23 -> 487515
submitted P1: YEZ 2024-04-15/09-11 -> 487516
submitted 

## 4. Status Check

In [27]:
status_rows = []
for pt in all_patients:
    pp0   = p0_paths(pt)
    for date_str, epoch_str, _, _ in get_epoch_model_entries(pt):
        pp   = p1_paths(pt, date_str, epoch_str)
        status_rows.append(dict(
            patient=pt, date_str=date_str, epoch_str=epoch_str,
            p0_done=pp0['success'].exists(),
            p1_done=pp['success'].exists(),
            p1_err=pp['error'].exists(),
            model_exists=(pp['out_dir'] / f'{pt}_epoch_model.tar').exists(),
        ))

status_df = pd.DataFrame(status_rows)
if not status_df.empty:
    print(f"P0 done: {status_df.groupby('patient')['p0_done'].first().sum()} / {status_df.patient.nunique()} patients")
    print(f"P1 done: {status_df.p1_done.sum()} / {len(status_df)} epoch fits")
    print(f"P1 errors: {status_df.p1_err.sum()}")
    display(status_df.groupby('patient').agg(
        n_epochs=('epoch_str','count'),
        p0_done=('p0_done','first'),
        p1_done=('p1_done','sum'),
        p1_err=('p1_err','sum'),
    ).reset_index())

P0 done: 20 / 21 patients
P1 done: 809 / 863 epoch fits
P1 errors: 0


,patient,n_epochs,p0_done,p1_done,p1_err
0,YEY,8,True,8,0
1,YEZ,25,True,25,0
2,YFA,54,False,0,0
3,YFB,48,True,48,0
4,YFC,50,True,50,0
5,YFD,33,True,33,0
6,YFE,20,True,20,0
7,YFF,58,True,58,0
8,YFG,22,True,22,0
9,YFI,39,True,39,0
